<a href="https://colab.research.google.com/github/sakshyambanjade/Algo-Trading-Bot/blob/develop/Random_Forest_Probability_of_Default_Model.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install pandas scikit-learn --quiet

In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score
from google.colab import files

In [ ]:
uploaded = files.upload()
file_name = list(uploaded.keys())[0]

Saving Task 3 and 4_Loan_Data.csv to Task 3 and 4_Loan_Data (1).csv


In [ ]:
data = pd.read_csv(file_name)
print("First 5 rows of the dataset:")
print(data.head())

First 5 rows of the dataset:
   customer_id  credit_lines_outstanding  loan_amt_outstanding  \
0      8153374                         0           5221.545193   
1      7442532                         5           1958.928726   
2      2256073                         0           3363.009259   
3      4885975                         0           4766.648001   
4      4700614                         1           1345.827718   

   total_debt_outstanding       income  years_employed  fico_score  default  
0             3915.471226  78039.38546               5         605        0  
1             8228.752520  26648.43525               2         572        1  
2             2027.830850  65866.71246               4         602        0  
3             2501.730397  74356.88347               5         612        0  
4             1768.826187  23448.32631               6         631        0  


In [ ]:
data = pd.read_csv(loan_data)
print("First 5 rows of the data:")
print(data.head())

First 5 rows of the data:
   customer_id  credit_lines_outstanding  loan_amt_outstanding  \
0      8153374                         0           5221.545193   
1      7442532                         5           1958.928726   
2      2256073                         0           3363.009259   
3      4885975                         0           4766.648001   
4      4700614                         1           1345.827718   

   total_debt_outstanding       income  years_employed  fico_score  default  
0             3915.471226  78039.38546               5         605        0  
1             8228.752520  26648.43525               2         572        1  
2             2027.830850  65866.71246               4         602        0  
3             2501.730397  74356.88347               5         612        0  
4             1768.826187  23448.32631               6         631        0  


In [ ]:
X = data.drop(columns=['customer_id', 'default'])
y = data['default']

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [ ]:
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

In [ ]:
rf = RandomForestClassifier(n_estimators=100, random_state=42)
rf.fit(X_train_scaled, y_train)

RandomForestClassifier(random_state=42)

In [ ]:
y_pred_prob = rf.predict_proba(X_test_scaled)[:, 1]
auc = roc_auc_score(y_test, y_pred_prob)
print(f"ROC-AUC Score: {auc:.3f}")

ROC-AUC Score: 1.000


In [ ]:
def expected_loss(loan_features, recovery_rate=0.10):
    """
    loan_features: dict with same keys as X columns
    recovery_rate: 0.1 = 10%
    """
    df = pd.DataFrame([loan_features])
    df_scaled = scaler.transform(df)
    pd_prob = rf.predict_proba(df_scaled)[:, 1][0]

    exposure = loan_features.get('loan_amt_outstanding', 0)
    exp_loss = pd_prob * exposure * (1 - recovery_rate)
    return exp_loss, pd_prob

In [ ]:
loan_example = {
    'credit_lines_outstanding': 5,
    'loan_amt_outstanding': 20000,
    'total_debt_outstanding': 25000,
    'income': 50000,
    'years_employed': 5,
    'fico_score': 700
}

In [ ]:
loss, pd_prob = expected_loss(loan_example)
print(f"Predicted Probability of Default: {pd_prob:.2f}")
print(f"Expected Loss: {loss:.2f}")

Predicted Probability of Default: 0.95
Expected Loss: 17100.00


In [ ]:
def expected_loss(loan_features, recovery_rate=0.10):
    df = pd.DataFrame([loan_features]) # Convert dictionary to DataFrame
    df_scaled = scaler.transform(df)
    pd_prob = rf.predict_proba(df_scaled)[:, 1][0]

    exposure = loan_features.get('loan_amt_outstanding', 0)
    exp_loss = pd_prob * exposure * (1 - recovery_rate)
    return exp_loss, pd_prob

In [ ]:
results = []
for idx, row in data.iterrows():
    # Prepare features as dict
    loan_features = row.drop(['customer_id', 'default']).to_dict()
    loss, pd_prob = expected_loss(loan_features)
    results.append({
        'customer_id': row['customer_id'],
        'predicted_default_prob': pd_prob,
        'expected_loss': loss
    })

In [ ]:
results_df = pd.DataFrame(results)
print(results_df.head())

   customer_id  predicted_default_prob  expected_loss
0    8153374.0                     0.0       0.000000
1    7442532.0                     1.0    1763.035853
2    2256073.0                     0.0       0.000000
3    4885975.0                     0.0       0.000000
4    4700614.0                     0.0       0.000000


In [ ]:
results_df.to_csv('loan_expected_loss.csv', index=False)
print("Expected loss results saved as 'loan_expected_loss.csv'")

Expected loss results saved as 'loan_expected_loss.csv'
